# Day 12: Week 2 综合复习 —— 阶段自测习题

> **范围**: Week 2 全部内容 —— SQL子查询/JOIN + Python函数/异常/文件IO + NumPy
> **建议用时**: 60-90 分钟
> **要求**: 每题独立完成，做完后回头扫一眼题目要求（防#42读题失误）

## Easy

**1. SQL —— 标量子查询**

基于 `../data/sales.csv`，用 SQL 找出 **订单总额(total)高于所有订单平均值** 的订单。
要求返回: order_id, customer_id, total。
（提示：标量子查询 `WHERE total > (SELECT AVG(total) FROM ...)`）

In [24]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

2026-06-14 17:53:03,977 [INFO] NumExpr defaulting to 8 threads.


Connecting to 'duckdb:///:memory:'

In [25]:
%%sql
SELECT * FROM (DESCRIBE SELECT * FROM '../data/sales.csv')

Running query in 'duckdb:///:memory:'

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
product,VARCHAR,YES,None,None,None
category,VARCHAR,YES,None,None,None
quantity,BIGINT,YES,None,None,None
price,BIGINT,YES,None,None,None
order_date,DATE,YES,None,None,None
country,VARCHAR,YES,None,None,None
total,BIGINT,YES,None,None,None


In [26]:
%%sql
CREATE OR REPLACE VIEW sales AS SELECT * FROM '../data/sales.csv'

Running query in 'duckdb:///:memory:'

Count


In [27]:
%%sql
SELECT order_id,
       customer_id,
       total
FROM sales
WHERE total > (SELECT AVG(total) FROM sales)

Running query in 'duckdb:///:memory:'

order_id,customer_id,total
O1000,C007,2598
O1006,C005,6495
O1014,C008,2997
O1015,C005,3996
O1018,C008,5997
O1020,C006,2997
O1021,C005,2995
O1022,C002,2598
O1023,C008,4995
O1027,C002,6495


**2. Python —— 安全除法 + 异常处理**

写函数 `safe_divide(a: float, b: float) -> float | None`:
- 正常返回 `a / b`
- `b == 0` 时 `logging.warning` 记录 "除零: a={a}, b={b}"，返回 `None`
- 输入不是数字时 `logging.warning` 记录异常信息，返回 `None`
- 测试: `safe_divide(10, 2)`, `safe_divide(5, 0)`, `safe_divide("x", 2)`

In [28]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def safe_divide(a:float, b:float) -> float | None:
    try:
        return a/b
    except ZeroDivisionError:
        logging.warning(f'除零：a={a}，b={b}')
        return None
    except TypeError as e:
        logging.warning(e)
        return None

In [29]:
print(safe_divide(10, 2))
print(safe_divide(5, 0))
print(safe_divide("x", 2))

2026-06-14 17:53:13,392 [WARNING] 除零：a=5，b=0
2026-06-14 17:53:13,398 [WARNING] unsupported operand type(s) for /: 'str' and 'int'


5.0
None
None


参考答案

问题: 用 try/except ZeroDivisionError 捕获除零，虽然结果正确，但更好的做法是前置检查 if b == 0，因为：

1. 题目明确说 "b == 0 时 logging.warning"，用 if 更直接表达意图
2. try/except 是「兜底」而不是「预期分支」，除零是预期内的情况，应该显式判断
3. 更关键的：except ZeroDivisionError 只会在运行时触发，但 b == 0 是可以在除法之前主动拦截的

In [57]:
def safe_divide(a: float, b: float) -> float | None:
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        logging.warning(f"输入不是数字: a={a}, b={b}")
        return None
    if b == 0:
        logging.warning(f"除零: a={a}, b={b}")
        return None
    return a / b

**3. NumPy —— 收益率筛选与占比**

给定某基金 20 天的日收益率(%):
```python
returns = np.array([0.5, -0.3, 1.2, 0.8, -0.5, 0.2, 1.5, -0.1, 0.6, 0.9,
                    -0.4, 0.3, 1.1, 0.7, -0.2, 0.4, 0.8, -0.6, 1.0, 0.5])
```
- 筛选出所有正收益，计算正收益的平均值
- 计算正收益天数占比（用布尔数组 `.mean()`）
- 把负收益替换为 0（原地修改），计算替换后的累计收益率（`cumsum`）
- 找出累计收益率最高的日期索引

In [30]:
import numpy as np

returns = np.array([0.5, -0.3, 1.2, 0.8, -0.5, 0.2, 1.5, -0.1, 0.6, 0.9,
                    -0.4, 0.3, 1.1, 0.7, -0.2, 0.4, 0.8, -0.6, 1.0, 0.5])

print(returns[returns > 0])
print(returns[returns > 0].mean())
print((returns > 0).mean())
returns[returns < 0] = 0
print(returns.cumsum())
print(returns.cumsum().argmax())

[0.5 1.2 0.8 0.2 1.5 0.6 0.9 0.3 1.1 0.7 0.4 0.8 1.  0.5]
0.75
0.7
[ 0.5  0.5  1.7  2.5  2.5  2.7  4.2  4.2  4.8  5.7  5.7  6.   7.1  7.8
  7.8  8.2  9.   9.  10.  10.5]
19


## Medium

**4. SQL —— LEFT JOIN + COALESCE**

假设有 `customers` 表（customer_id, name, country）和 `sales` 表。
写 SQL 返回 **每个客户的名字、国家、以及订单总额**。
要求:
- 没有订单的客户也要显示（总额显示为 0，不是 NULL）
- 用 `COALESCE` 处理 NULL
- 结果按订单总额降序排列

（提示：LEFT JOIN + GROUP BY + `COALESCE(SUM(total), 0)`）

In [32]:
%%sql
CREATE OR REPLACE VIEW customers AS SELECT * FROM '../data/customers.csv'

Running query in 'duckdb:///:memory:'

Count


In [33]:
%%sql
SELECT * FROM customers

Running query in 'duckdb:///:memory:'

customer_id,name,country,signup_date
C001,Alice,France,2023-01-15
C002,Bob,Germany,2023-02-20
C003,Charlie,France,2023-03-10
C004,David,US,2023-04-05
C005,Eva,US,2023-05-12
C006,Frank,China,2023-06-18
C007,Grace,Germany,2023-07-22
C008,Henry,UK,2023-08-30
C009,Ivy,Japan,2023-09-14
C010,Jack,Canada,2023-10-01


In [35]:
%%sql
SELECT
    c.name,
    c.country,
    COALESCE(SUM(s.total), 0) AS total 
FROM customers c 
LEFT JOIN sales s 
    ON c.customer_id = s.customer_id
GROUP BY c.name, c.country
ORDER BY total DESC

Running query in 'duckdb:///:memory:'

name,country,total
Frank,China,198806
Alice,France,184001
Henry,UK,164484
Charlie,France,160212
David,US,144095
Grace,Germany,143917
Bob,Germany,140967
Eva,US,131234
Ivy,Japan,0
Jack,Canada,0


**5. Python —— 装饰器计时**

写装饰器 `@timer`:
- 用 `logging.info` 记录 "函数 {name} 耗时 {seconds:.4f} 秒"
- 必须正确处理被装饰函数的参数（`*args, **kwargs`）和返回值
- 装饰一个模拟慢操作的函数 `slow_add(a, b)`（用 `time.sleep(0.1)`），测试是否正常返回 `a+b` 且日志正确

In [37]:
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"函数{func.__name__}耗时{time.time() - start:.4f}秒")
        return result
    return wrapper

@timer
def slow_add(a, b):
    time.sleep(0.1)
    return a+b

slow_add(1, 2)

函数slow_add耗时0.1005秒


3

参考答案

问题: 题目明确要求「用 logging.info 记录」，你用了 print。虽然输出看起来一样，但：

print 没有级别、没有时间戳、不能输出到文件

logging.info 是正经代码的标准做法，和 print 有本质区别

In [58]:
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        logging.info(f"函数 {func.__name__} 耗时 {elapsed:.4f} 秒")
        return result
    return wrapper

**6. Python + NumPy + 文件IO —— 列统计**

写函数 `column_stats(csv_path, column_name)`:
- 用 `csv.DictReader` 读取 CSV
- 把指定列的所有值转成 NumPy 数组（float 类型）
- 坏行（转不了数字的）用 `logging.warning` 记录并跳过
- 文件不存在时返回 `None` 并 `logging.warning`
- 返回一个 dict: `{"count": ..., "mean": ..., "std": ..., "max": ..., "min": ...}`
- 用 `../data/sales.csv` 测试，分析 `total` 列

In [55]:
import csv
import logging
from pathlib import Path
import numpy as np


def column_status(csv_path, column_name):
    path = Path(csv_path)
    if not path.exists():
        logging.warning(f"文件不存在：{path}")
        return None
    lst = []
    with path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row_idx, row in enumerate(reader, start=2):
            raw_value = row[column_name].strip()
            try:
                num = float(raw_value)
                lst.append(num)
            except ValueError:
                logging.warning(f"第{row_idx}行列{column_name}值{raw_value}无法转化为float，已跳过")
    arr = np.array(lst, dtype=float)
    count = arr.size
    
    if count == 0:
        stats_dict = {
            "count": 0,
            "mean": np.nan,
            "std": np.nan,
            "max": np.nan,
            "min": np.nan
        }
    else:
        stats_dict = {
            "count": count,
            "mean": float(arr.mean()),
            "std": float(arr.std()),
            "max": float(arr.max()),
            "min": float(arr.min())
        }
    return stats_dict

In [56]:
print(column_status('../data/sales.csv', 'total'))

{'count': 500, 'mean': 2535.432, 'std': 2423.819831871998, 'max': 9995.0, 'min': 99.0}


**7. SQL —— 两步聚合（派生表）**

基于 `../data/sales.csv`:
- 第一步：按 `country` 分组，计算每个国家的订单数和总销售额
- 第二步：从第一步的结果中，找出 **总销售额前 3 名** 的国家

要求用 **派生表**（`FROM (SELECT ... GROUP BY ...) AS t`）实现，不能用窗口函数。
返回: country, order_count, total_sales

In [ ]:
%%sql
SELECT country,
       order_count,
       total_sales
FROM(
    SELECT country,
           COUNT(*) AS order_count,
           SUM(total) AS total_sales
    FROM sales
    GROUP BY country
) AS t
ORDER BY total_sales DESC
LIMIT 3;

Running query in 'duckdb:///:memory:'

country,order_count,total_sales
UK,197,534817
US,125,300613
France,80,208165


## Hard

**8. Python + NumPy + 文件IO —— 品类分析管道**

写函数 `category_analysis(csv_path, category_name)`:
1. 读取 `../data/sales.csv`
2. 筛选出指定品类（如 `"Food"` 或 `"Electronics"`）的所有记录
3. 把 `total` 列转成 NumPy 数组，坏数据跳过 + `logging.warning`
4. 计算该品类的：订单数、总销售额、平均订单额、标准差、中位数
5. 找出该品类中订单额 **超过该品类平均值** 的所有订单索引（在筛选后数组中的位置）
6. 把结果写成 JSON 文件 `{category_name}_report.json`

要求全程异常安全：文件不存在、空文件、无该品类记录 都不能崩。
测试品类: `"Electronics"`（或你数据里有的品类名，先用 `DESCRIBE` 或 `SELECT DISTINCT category` 确认）

In [46]:
import csv
import json
import logging
from pathlib import Path
import numpy as np

def category_analysis(csv_path, category_name):
    path = Path(csv_path)
    if not path.exists():
        logging.warning(f"文件不存在：{path}")
        return None

    category_totals = []
    row_index = 2  

    try:
        with path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            need_cols = {"category", "total"}
            if not need_cols.issubset(reader.fieldnames):
                logging.warning("CSV缺少category或total字段")
                return None

            for row in reader:
                if row["category"].strip() != category_name.strip():
                    row_index += 1
                    continue

                raw_total = row["total"].strip()
                try:
                    val = float(raw_total)
                    category_totals.append(val)
                except ValueError:
                    logging.warning(f"第{row_index}行 total值'{raw_total}'无法转为数值，跳过")
                row_index += 1

    except Exception as e:
        logging.warning(f"读取文件异常：{str(e)}")
        return None


    arr = np.array(category_totals, dtype=np.float64)
    order_count = arr.size

    if order_count == 0:
        report = {
            "category": category_name,
            "order_count": 0,
            "total_sales": 0.0,
            "avg_order": None,
            "std_order": None,
            "median_order": None,
            "above_avg_indexes": []
        }
    else:
        avg = arr.mean()
        above_idx = np.where(arr > avg)[0].tolist()
        report = {
            "category": category_name,
            "order_count": int(order_count),
            "total_sales": float(arr.sum()),
            "avg_order": float(avg),
            "std_order": float(arr.std()),
            "median_order": float(np.median(arr)),
            "above_avg_indexes": above_idx
        }


    out_filename = f"{category_name}_report.json"
    try:
        with open(out_filename, "w", encoding="utf-8") as fw:
            json.dump(report, fw, ensure_ascii=False, indent=2)
        print(f"报告已生成：{out_filename}")
    except Exception as e:
        logging.warning(f"写入JSON失败：{str(e)}")

    return report


In [48]:
print(category_analysis("../data/sales.csv", "Computer"))

报告已生成：Computer_report.json
{'category': 'Computer', 'order_count': 160, 'total_sales': 363933.0, 'avg_order': 2274.58125, 'std_order': 2176.0688134566053, 'median_order': 1495.0, 'above_avg_indexes': [5, 7, 8, 10, 12, 13, 24, 32, 33, 35, 37, 39, 40, 42, 44, 45, 46, 47, 48, 49, 52, 55, 68, 69, 70, 73, 74, 75, 79, 81, 82, 83, 88, 89, 99, 101, 104, 107, 108, 110, 114, 115, 117, 118, 119, 121, 122, 124, 125, 127, 130, 136, 139, 141, 147, 156, 157, 158, 159]}


**9. SQL —— 综合：多表 + 子查询 + HAVING**

基于 `../data/sales.csv`（把它当订单表），写 SQL 找出 **优质客户**:
- 定义：下单次数 > 2 次 **且** 平均订单额 > **所有客户的平均订单额**
- 返回: customer_id, order_count, avg_order, total_spent
- 按 total_spent 降序

要求:
- 用 `GROUP BY` + `HAVING` 做次数过滤
- 用标量子查询算「所有客户的平均订单额」
- HAVING 里写聚合函数本身，不要用 SELECT 别名（防#15）

In [49]:
%%sql

SELECT customer_id,
       COUNT(*) AS order_count,
       AVG(total) AS avg_order,
       SUM(total) AS total_spent
FROM sales
GROUP BY customer_id
HAVING 
       COUNT(*) > 2
       AND AVG(total) > (
        SELECT AVG(cust_avg)
        FROM (
            SELECT AVG(total) AS cust_avg
            FROM sales
            GROUP BY customer_id
        ) AS customer_avg_table
       )
ORDER BY total_spent DESC;

Running query in 'duckdb:///:memory:'

customer_id,order_count,avg_order,total_spent
C006,62,3206.548387096774,198806
C001,71,2591.56338028169,184001
C002,46,3064.5,140967


**10. Python —— 数据管道 + 边界测试**

写函数 `process_pipeline(csv_path)`，完成完整数据管道:

**阶段1 —— 读取**:
- 用 `pathlib` 检查文件存在性，不存在返回 `None` + `logging.warning`
- 用 `csv.DictReader` 读取

**阶段2 —— 清洗**:
- 只保留 `quantity`（转int）和 `total`（转float）都成功的行
- 坏行 `logging.warning` 记录并跳过
- 空文件时返回空列表（不崩）

**阶段3 —— NumPy分析**:
- 把 `quantity` 和 `total` 分别转成两个 NumPy 数组
- 计算 `total` 的: count, sum, mean, std, max, min
- 计算 `quantity` 的: count, sum, mean
- 计算单价（`total / quantity`）的 NumPy 数组，求平均单价

**阶段4 —— 返回报告**:
返回 dict:
```python
{
    "record_count": int,
    "bad_count": int,
    "total_stats": {"sum": float, "mean": float, "std": float, "max": float, "min": float},
    "quantity_stats": {"sum": int, "mean": float},
    "avg_unit_price": float
}
```

**边界测试**（在 cell 末尾列出）:
- 至少测试 3 种边界情况（如：不存在的文件、空文件、全坏数据的文件）
- 说明每种情况的预期行为和实际结果

In [51]:
import csv
import logging
from pathlib import Path
import numpy as np

def process_pipeline(csv_path):
    file_path = Path(csv_path)
    if not file_path.exists():
        logging.warning(f"文件 {csv_path} 不存在")
        return None
    
    valid_quantity = []
    valid_total = []
    bad_count = 0

    try:
        with open(file_path, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row_idx, row in enumerate(reader, start=2):
                q_raw = row.get("quantity", "").strip()
                t_raw = row.get("total", "").strip()
                try:
                    q_val = int(q_raw)
                    t_val = float(t_raw)
                    valid_quantity.append(q_val)
                    valid_total.append(t_val)
                except (ValueError, TypeError):
                    bad_count += 1
                    logging.warning(f"第{row_idx}行数据清洗失败，quantity:{q_raw}, total:{t_raw}，已跳过")
    except Exception as e:
        logging.warning(f"文件读取异常：{str(e)}")
        return None

    q_arr = np.array(valid_quantity, dtype=np.int64)
    t_arr = np.array(valid_total, dtype=np.float64)
    record_count = q_arr.size

    total_stats = {
        "sum": float(t_arr.sum()),
        "mean": float(t_arr.mean()) if record_count > 0 else 0.0,
        "std": float(t_arr.std()) if record_count > 0 else 0.0,
        "max": float(t_arr.max()) if record_count > 0 else 0.0,
        "min": float(t_arr.min()) if record_count > 0 else 0.0
    }

    quantity_stats = {
        "sum": int(q_arr.sum()),
        "mean": float(q_arr.mean()) if record_count > 0 else 0.0
    }

    if record_count == 0:
        avg_unit_price = 0.0
    else:
        unit_price_arr = t_arr / q_arr
        avg_unit_price = float(unit_price_arr.mean())

    result_dict = {
        "record_count": record_count,
        "bad_count": bad_count,
        "total_stats": total_stats,
        "quantity_stats": quantity_stats,
        "avg_unit_price": avg_unit_price
    }
    return result_dict



In [53]:
print("=== 测试1：不存在文件 ===")
res1 = process_pipeline("no_exist.csv")
print("返回结果：", res1)

   
print("\n=== 测试2：空数据文件 ===")
res2 = process_pipeline("empty.csv")
print("返回结果：", res2)
    

print("\n=== 测试3：全部脏数据文件 ===")
res3 = process_pipeline("all_bad.csv")
print("返回结果：", res3)


print("\n=== 测试4：正常数据文件 ===")
res4 = process_pipeline("../data/sales.csv")
print("返回结果：", res4)

2026-06-14 22:00:52,281 [WARNING] 文件 no_exist.csv 不存在
2026-06-14 22:00:52,284 [WARNING] 文件 empty.csv 不存在
2026-06-14 22:00:52,286 [WARNING] 文件 all_bad.csv 不存在


=== 测试1：不存在文件 ===
返回结果： None

=== 测试2：空数据文件 ===
返回结果： None

=== 测试3：全部脏数据文件 ===
返回结果： None

=== 测试4：正常数据文件 ===
返回结果： {'record_count': 500, 'bad_count': 0, 'total_stats': {'sum': 1267716.0, 'mean': 2535.432, 'std': 2423.819831871998, 'max': 9995.0, 'min': 99.0}, 'quantity_stats': {'sum': 1484, 'mean': 2.968}, 'avg_unit_price': 863.2}


参考答案

问题: 你测试了 4 种情况，但「空文件」和「全坏数据」这两个测试文件不存在，所以实际测的是「文件不存在」场景，而不是「空文件」和「全坏数据」。

In [59]:
import csv

# 1. 创建空文件（只有表头，没有数据行）
with open("empty.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["quantity", "total"])
    writer.writeheader()

# 2. 创建全坏数据文件
with open("all_bad.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["quantity", "total"])
    writer.writeheader()
    writer.writerows([
        {"quantity": "abc", "total": "xyz"},
        {"quantity": "", "total": "bad"},
    ])

# 3. 现在测试
print("=== 测试2：空文件 ===")
print(process_pipeline("empty.csv"))  # 预期: record_count=0, bad_count=0

print("=== 测试3：全坏数据 ===")
print(process_pipeline("all_bad.csv"))  # 预期: record_count=0, bad_count=2

2026-06-14 22:07:21,356 [WARNING] 第2行数据清洗失败，quantity:abc, total:xyz，已跳过
2026-06-14 22:07:21,358 [WARNING] 第3行数据清洗失败，quantity:, total:bad，已跳过


=== 测试2：空文件 ===
{'record_count': 0, 'bad_count': 0, 'total_stats': {'sum': 0.0, 'mean': 0.0, 'std': 0.0, 'max': 0.0, 'min': 0.0}, 'quantity_stats': {'sum': 0, 'mean': 0.0}, 'avg_unit_price': 0.0}
=== 测试3：全坏数据 ===
{'record_count': 0, 'bad_count': 2, 'total_stats': {'sum': 0.0, 'mean': 0.0, 'std': 0.0, 'max': 0.0, 'min': 0.0}, 'quantity_stats': {'sum': 0, 'mean': 0.0}, 'avg_unit_price': 0.0}
